In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
# Cell 1 — Classification notebook install (NO unsloth)
!pip install -q pyreft
!pip install -q transformers==4.45.1 datasets peft accelerate
!pip install -q evaluate bert-score scikit-learn
!pip install -q wandb matplotlib seaborn pandas pyarrow
!pip install -q presidio-analyzer presidio-anonymizer spacy
!python -m spacy download en_core_web_lg -q

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
# ── Suppress noise ────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
#-------------------------------------------------------------

import transformers
import pyreft
from peft import get_peft_model, LoraConfig, TaskType
from datasets import load_dataset
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
import evaluate
import wandb
import torch
from pyreft import (
    ReftConfig,
    LoreftIntervention,
    ReftModel,
    get_reft_model
)

print("transformers:", transformers.__version__)   # must be 4.45.1
print("pyreft:      imported" )
print("CUDA:        ", torch.cuda.is_available())
print("GPU:         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("─" * 35)
print("All imports successful")

nnsight is not detected. Please install via 'pip install nnsight' for nnsight backend.


E0000 00:00:1774593142.432614     151 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774593142.490747     151 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774593142.948114     151 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774593142.948139     151 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774593142.948142     151 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774593142.948144     151 computation_placer.cc:177] computation placer already registered. Please check linka

transformers: 4.45.1
pyreft:      imported
CUDA:         True
GPU:          Tesla T4
───────────────────────────────────
All imports successful


In [ ]:
!git clone https://github.com/chetnapriyadarshini/Thesis_PeFT

In [8]:
!git config --global user.email "chetna.priya@gmail.com"
!git config --global user.name "Chetna Priyadarshini"

In [8]:
!git checkout --theirs generation/train_lora_llama.py
!git add generation/train_lora_llama.py
!git commit -m "resolve merge conflict - keep remote version of train_lora_llama.py"

Updated 1 path from the index
[main e6f2645] resolve merge conflict - keep remote version of train_lora_llama.py


In [9]:
!head -50 generation/train_lora_llama.py

# =============================================================================
# generation/train_lora_llama.py
# Llama 3.2 3B Instruct + LoRA — Empathetic Dialogue Generation
#
# Key differences from classification:
#   - Causal LM task (generation) vs sequence classification
#   - 4-bit quantization (QLoRA) to fit 3B model on T4 16GB
#   - SFTTrainer with instruction template + loss masking on prompt tokens
#   - LoRA targets attention + MLP projections (Llama architecture)
#   - Evaluation: BERTScore (semantic similarity) — EPIC/LLM-as-Judge run separately
# =============================================================================

#Package.                        Used by
#transformersAutoTokenizer,      AutoModelForCausalLM, set_seed
#datasets.                       load_from_diskpeftLoraConfig, TaskType, get_peft_modelaccelerateRequired internally by transformers Trainer for GPU trainingtrlSFTTrainer, SFTConfigbitsandbytespaged_adamw_8bit optimiserwandbexperiment tracking
#pe

In [5]:
%cd /kaggle/working/Thesis_PeFT
!git pull origin main --allow-unrelated-histories
!ls

/kaggle/working/Thesis_PeFT
From https://github.com/chetnapriyadarshini/Thesis_PeFT
 * branch            main       -> FETCH_HEAD
hint: You have divergent branches and need to specify how to reconcile them.
hint: You can do so by running one of the following commands sometime before
hint: your next pull:
hint: 
hint:   git config pull.rebase false  # merge (the default strategy)
hint:   git config pull.rebase true   # rebase
hint:   git config pull.ff only       # fast-forward only
hint: 
hint: You can replace "git config" with "git config --global" to set a default
hint: preference for all repositories. You can also pass --rebase, --no-rebase,
hint: or --ff-only on the command line to override the configured default per
hint: invocation.
fatal: Need to specify how to reconcile divergent branches.
classification	data	    LICENSE    notes	    README.md	      results
config.py	generation  notebooks  __pycache__  requirements.txt  wandb


In [ ]:
%run config.py

In [5]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Load token from Kaggle secrets
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

login(token=hf_token)
print("HuggingFace login successful ✅")

HuggingFace login successful ✅


In [ ]:
from datasets import load_dataset
import pandas as pd

# Load SWMH
print("Loading SWMH...")
swmh = load_dataset("AIMH/SWMH")
print(swmh)

In [ ]:
import pandas as pd

# Combine all splits
dfs = []
for split in swmh:
    df = swmh[split].to_pandas()
    df["split"] = split
    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)

print("Columns:", combined.columns.tolist())
print(f"\nTotal rows: {len(combined)}")
print(f"\nLabel distribution (all splits combined):")
print(combined["label"].value_counts().sort_index())
print(f"\nLabel distribution by split:")
print(combined.groupby(["split", "label"]).size().unstack(fill_value=0))
print(f"\nSample rows:")
combined.head(10)

In [ ]:
import matplotlib.pyplot as plt

# Current SWMH class sizes
label_counts = combined["label"].value_counts().sort_index()
print("Current SWMH class distribution:")
print(label_counts)
print(f"\nSmallest class: {label_counts.min()} ({label_counts.idxmin()})")
print(f"Largest class:  {label_counts.max()} ({label_counts.idxmax()})")
print(f"Imbalance ratio: {label_counts.max() / label_counts.min():.2f}x")

print("\n--- Key question for IMDB ---")
print(f"If we match smallest class (bipolar): ~{label_counts.min()} IMDB samples")
print(f"If we match mean:                     ~{int(label_counts.mean())} IMDB samples")
print(f"If we match largest (depression):     ~{label_counts.max()} IMDB samples")

# Visual
label_counts.plot(kind="bar", figsize=(8, 4), title="SWMH Class Distribution")
plt.ylabel("Count")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
offmychest_samples = combined[combined["label"] == "self.offmychest"].sample(10, random_state=SEED)

for i, row in offmychest_samples.iterrows():
    print(f"--- Sample {i} ---")
    print(row["text"][:500])  # first 500 chars
    print()

In [ ]:
import re

# Keywords that suggest clinical mental health content
mh_keywords = [
    "suicide", "suicidal", "self harm", "self-harm", "overdose",
    "depression", "depressed", "anxiety", "bipolar", "schizophrenia",
    "mental illness", "mental health",
    "antidepressant", "panic attack", "ptsd", "trauma",
    "eating disorder", "anorexia", "bulimia", "psychosis", "hallucin"
]

pattern = "|".join(mh_keywords)

offmychest_df = combined[combined["label"] == "self.offmychest"].copy()
offmychest_df["has_mh_keyword"] = offmychest_df["text"].str.lower().str.contains(pattern, na=False)

flagged = offmychest_df[offmychest_df["has_mh_keyword"]]
clean   = offmychest_df[~offmychest_df["has_mh_keyword"]]

print(f"Total offmychest:        {len(offmychest_df)}")
print(f"Flagged (MH keywords):   {len(flagged)} ({len(flagged)/len(offmychest_df)*100:.1f}%)")
print(f"Clean (no MH keywords):  {len(clean)} ({len(clean)/len(offmychest_df)*100:.1f}%)")

print("\n--- 5 random flagged samples ---")
for i, row in flagged.sample(5, random_state=SEED).iterrows():
    print(f"\n[{i}] {row['text'][:400]}")
    print("---")

In [ ]:
# Check for nulls and empty texts
print("=== Null / Empty Check ===")
for label in combined["label"].unique():
    subset = combined[combined["label"] == label]
    nulls = subset["text"].isna().sum()
    empty = (subset["text"].str.strip() == "").sum()
    deleted = (subset["text"].str.strip() == "[deleted]").sum()
    print(f"{label}: nulls={nulls}, empty={empty}, [deleted]={deleted}")

# Check text length distribution
print("\n=== Text Length Distribution ===")
combined["text_len"] = combined["text"].str.split().str.len()
print(combined.groupby("label")["text_len"].describe().round(1))

# Check for very short texts (under 10 words)
print("\n=== Very short texts (< 10 words) per class ===")
short = combined[combined["text_len"] < 10]
print(short["label"].value_counts())

In [ ]:
import warnings
warnings.filterwarnings('ignore')
%cd /kaggle/working/Thesis_PeFT
!git pull origin main
!python data/prepare_data_classification.py

In [ ]:
# Verify what was saved
import os
for f in os.listdir("/kaggle/working/data/classification"):
    print(f)

In [ ]:
%cd /kaggle/working/Thesis_PeFT
!git pull origin main
!python data/prepare_data_generation.py

In [17]:
!git fetch origin main
!git reset --hard origin/main

From https://github.com/chetnapriyadarshini/Thesis_PeFT
 * branch            main       -> FETCH_HEAD
HEAD is now at ac8e5fe Update WanDB config to num_steps as well


In [18]:
%cd /kaggle/working/Thesis_PeFT
!git pull origin main

/kaggle/working/Thesis_PeFT
From https://github.com/chetnapriyadarshini/Thesis_PeFT
 * branch            main       -> FETCH_HEAD
Already up to date.


In [4]:
from kaggle_secrets import UserSecretsClient
import wandb
secrets = UserSecretsClient()
wandb.login(key=secrets.get_secret("WANDB_API_KEY"))

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: chetna-priya (chetna-priya-liverpool-john-moores-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [16]:
!python classification/train_lora.py

E0000 00:00:1774264735.440794     509 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774264735.448333     509 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774264735.468141     509 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774264735.468204     509 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774264735.468212     509 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774264735.468218     509 computation_placer.cc:177] computation placer already registered. Please check linka

In [14]:
%cd /kaggle/working/Thesis_PeFT
!git pull origin main

/kaggle/working/Thesis_PeFT
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 8 (delta 6), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 999 bytes | 333.00 KiB/s, done.
From https://github.com/chetnapriyadarshini/Thesis_PeFT
 * branch            main       -> FETCH_HEAD
   0d7a8cc..bfa8841  main       -> origin/main
Updating 0d7a8cc..bfa8841
Fast-forward
 generation/train_lora_llama.py | 6 +++---
 1 file changed, 3 insertions(+), 3 deletions(-)


In [16]:
%cd /kaggle/working/Thesis_PeFT
!python classification/train_reft.py

/kaggle/working/Thesis_PeFT
E0000 00:00:1774350937.483867     784 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774350937.492610     784 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774350937.516226     784 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774350937.516256     784 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774350937.516261     784 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774350937.516264     784 computation_placer.cc:177] computation placer already re

In [7]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=5)
for name, module in model.named_modules():
    print(name)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



distilbert
distilbert.embeddings
distilbert.embeddings.word_embeddings
distilbert.embeddings.position_embeddings
distilbert.embeddings.LayerNorm
distilbert.embeddings.dropout
distilbert.transformer
distilbert.transformer.layer
distilbert.transformer.layer.0
distilbert.transformer.layer.0.attention
distilbert.transformer.layer.0.attention.dropout
distilbert.transformer.layer.0.attention.q_lin
distilbert.transformer.layer.0.attention.k_lin
distilbert.transformer.layer.0.attention.v_lin
distilbert.transformer.layer.0.attention.out_lin
distilbert.transformer.layer.0.sa_layer_norm
distilbert.transformer.layer.0.ffn
distilbert.transformer.layer.0.ffn.dropout
distilbert.transformer.layer.0.ffn.lin1
distilbert.transformer.layer.0.ffn.lin2
distilbert.transformer.layer.0.ffn.activation
distilbert.transformer.layer.0.output_layer_norm
distilbert.transformer.layer.1
distilbert.transformer.layer.1.attention
distilbert.transformer.layer.1.attention.dropout
distilbert.transformer.layer.1.attention.q

In [12]:
import inspect
from pyvene.models import modeling_utils
print(inspect.getsource(modeling_utils.get_module_hook))

def get_module_hook(model, representation, backend="native") -> nn.Module:
    """Render the intervening module with a hook."""
    if (
        get_internal_model_type(model) in type_to_module_mapping and
        representation.component
        in type_to_module_mapping[get_internal_model_type(model)]
    ):
        type_info = type_to_module_mapping[get_internal_model_type(model)][
            representation.component
        ]
        parameter_name = type_info[0]
        hook_type = type_info[1]
        if "%s" in parameter_name and representation.moe_key is None:
            # we assume it is for the layer.
            parameter_name = parameter_name % (representation.layer)
        elif "%s" in parameter_name and representation.moe_key is not None:
            parameter_name = parameter_name % (
                int(representation.layer),
                int(representation.moe_key),
            )
    else:
        parameter_name = ".".join(representation.component.split(".")[:-1]

In [19]:
print(tokenized["train"].features)
print(tokenized["train"][0])

NameError: name 'tokenized' is not defined

In [22]:
from datasets import load_from_disk
from transformers import AutoTokenizer
import numpy as np

dataset = load_from_disk("/kaggle/working/data/classification")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

lengths = [len(tokenizer(t, truncation=False)["input_ids"]) 
           for t in dataset["train"]["text"]]

print(f"Max length:    {np.max(lengths)}")
print(f"Mean length:   {np.mean(lengths):.1f}")
print(f"Median length: {np.median(lengths):.1f}")
print(f"90th pct:      {np.percentile(lengths, 90):.1f}")
print(f"95th pct:      {np.percentile(lengths, 95):.1f}")
print(f"99th pct:      {np.percentile(lengths, 99):.1f}")
print(f"% over 256:    {np.mean(np.array(lengths) > 256)*100:.1f}%")
print(f"% over 512:    {np.mean(np.array(lengths) > 512)*100:.1f}%")

Token indices sequence length is longer than the specified maximum sequence length for this model (960 > 512). Running this sequence through the model will result in indexing errors


Max length:    7028
Mean length:   216.6
Median length: 135.0
90th pct:      492.0
95th pct:      686.0
99th pct:      1282.3
% over 256:    27.5%
% over 512:    9.3%


In [2]:
!pip install -q transformers datasets peft accelerate trl
!pip install -q bitsandbytes evaluate bert-score
!pip install -q wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 10.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.0 MB/s eta 0:00:00


In [12]:
%cd /kaggle/working/Thesis_PeFT
!git pull origin main

/kaggle/working/Thesis_PeFT
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 612 bytes | 612.00 KiB/s, done.
From https://github.com/chetnapriyadarshini/Thesis_PeFT
 * branch            main       -> FETCH_HEAD
   8896313..0d7a8cc  main       -> origin/main
Updating 8896313..0d7a8cc
Fast-forward
 generation/train_lora_llama.py | 6 ++----
 1 file changed, 2 insertions(+), 4 deletions(-)


In [3]:
%cd /kaggle/working/Thesis_PeFT
!python generation/train_lora_llama.py

/kaggle/working/Thesis_PeFT
Config loaded |  BASE_DIR: /kaggle/working
Config loaded |  BASE_DIR: /kaggle/working

── Loading generation dataset ──────────────────────────────────────────
DatasetDict({
    train: Dataset({
        features: ['prompt', 'response', 'emotion'],
        num_rows: 45213
    })
    val: Dataset({
        features: ['prompt', 'response', 'emotion'],
        num_rows: 9689
    })
    test: Dataset({
        features: ['prompt', 'response', 'emotion'],
        num_rows: 9689
    })
})

Sample prompt:
[anticipating] I have big trip coming up to <LOCATION>. I have never been before so I am getting a little antsy. | sounds like it going to be an interesting trip

Sample response:
I cant wait to see the beaches and the hills. Im having a hard time trying to concentrate on work.

── Formatting dataset as instruction template ───────────────────────────
DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 45213
    })
    val: Dataset(

In [1]:
from datasets import load_from_disk
ds = load_from_disk("/kaggle/working/data/generation")
print(ds["train"].column_names)
print(ds["train"][0])

['prompt', 'response', 'emotion']
{'prompt': '[anticipating] I have big trip coming up to <LOCATION>. I have never been before so I am getting a little antsy. | sounds like it going to be an interesting trip', 'response': 'I cant wait to see the beaches and the hills. Im having a hard time trying to concentrate on work.', 'emotion': 'anticipating'}


In [4]:
!pip install -q transformers datasets peft accelerate
!pip install -q trl bitsandbytes
!pip install -q wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 9.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.2 MB/s eta 0:00:00:00:0100:01


In [ ]:
%cd /kaggle/working/Thesis_PeFT
!python generation/train_lora_llama.py

/kaggle/working/Thesis_PeFT
Config loaded |  BASE_DIR: /kaggle/working
Config loaded |  BASE_DIR: /kaggle/working

── Loading generation dataset ──────────────────────────────────────────
DatasetDict({
    train: Dataset({
        features: ['prompt', 'response', 'emotion'],
        num_rows: 45213
    })
    val: Dataset({
        features: ['prompt', 'response', 'emotion'],
        num_rows: 9689
    })
    test: Dataset({
        features: ['prompt', 'response', 'emotion'],
        num_rows: 9689
    })
})

Sample prompt:
[anticipating] I have big trip coming up to <LOCATION>. I have never been before so I am getting a little antsy. | sounds like it going to be an interesting trip

Sample response:
I cant wait to see the beaches and the hills. Im having a hard time trying to concentrate on work.

── Formatting dataset as instruction template ───────────────────────────
DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 45213
    })
    val: Dataset(